# UNSW-NB15 Data Preprocessing

This notebook only handles:
- loading the dataset
- train/test split
- removing leakage columns
- identifying categorical columns
- encoding categorical features
- saving the fitted preprocessor and processed train/test arrays

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
# Notebook is inside SIH_1/notebook, so ../dataset points to SIH_1/dataset
DATASET_PATH = "../dataset/UNSW_NB15_testing-set.parquet"
MODELS_DIR = "../models"
PROCESSED_DIR = "../dataset/processed"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [3]:
dataset = pd.read_parquet(DATASET_PATH)

dataset.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.000011,udp,-,INT,2,0,496,0,90909.09375,180363632.0,...,0,0,1,1,0,0,0,0,Normal,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.00000,881000000.0,...,0,0,1,1,0,0,0,0,Normal,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.00000,854400000.0,...,0,0,1,1,0,0,0,0,Normal,0
3,0.000006,udp,-,INT,2,0,900,0,166666.65625,600000000.0,...,0,0,2,1,0,0,0,0,Normal,0
4,0.000010,udp,-,INT,2,0,2126,0,100000.00000,850400000.0,...,0,0,2,1,0,0,0,0,Normal,0


In [4]:
# Use label as the binary target when available.
target_column = "label" if "label" in dataset.columns else dataset.columns[-1]

X = dataset.drop(columns=[target_column])
y = dataset[target_column]

# attack_cat reveals the attack type and can leak the binary label.
if "attack_cat" in X.columns:
    X = X.drop(columns=["attack_cat"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (82332, 34)
y shape: (82332,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (65865, 34)
Test shape: (16467, 34)


In [6]:
# Check missing values
missing = X_train.isnull().sum()
missing[missing > 0]

Series([], dtype: int64)

In [7]:
# UNSW-NB15 may store categorical columns as object, category, or pandas string dtype.
categorical_columns = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Categorical columns:", categorical_columns)

Categorical columns: ['proto', 'service', 'state']


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns
        )
    ],
    remainder="passthrough"
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (65865, 182)
Processed test shape: (16467, 182)


In [9]:
# Save preprocessor in the models folder
joblib.dump(preprocessor, os.path.join(MODELS_DIR, "preprocessor.pkl"))

# Save processed arrays for the training/testing notebooks
np.save(os.path.join(PROCESSED_DIR, "X_train.npy"), X_train_processed)
np.save(os.path.join(PROCESSED_DIR, "X_test.npy"), X_test_processed)
np.save(os.path.join(PROCESSED_DIR, "y_train.npy"), np.asarray(y_train))
np.save(os.path.join(PROCESSED_DIR, "y_test.npy"), np.asarray(y_test))

print("Preprocessing files saved.")

Preprocessing files saved.
